# Shared memory: tiled matrix transpose

A transpose changes the layout of a matrix. In the direct kernel, each warp reads adjacent input values, but its lanes write to output addresses separated by the matrix width. The tiled kernel uses shared memory to rearrange each tile before writing it, so both global-memory reads and writes are contiguous across lanes. The program validates both GPU outputs against a CPU transpose and reports average kernel times.

## Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → GPU**. This cell displays the selected device and creates a folder for the CUDA source files.

In [1]:
!nvidia-smi
!nvcc --version
!mkdir -p code


Sat Nov 14 07:51:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## CUDA error and timing support

The error-checking macro is independent of the timer. `GetCounter()` returns elapsed milliseconds for the recorded GPU work.

In [2]:
%%writefile code/cuda_check.cuh
#ifndef CUDA_CHECK_CUH
#define CUDA_CHECK_CUH
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#define CUDA_CHECK(call) do { \
    cudaError_t status = (call); \
    if (status != cudaSuccess) { \
        std::fprintf(stderr, "%s:%d: %s\n", __FILE__, __LINE__, \
                     cudaGetErrorString(status)); \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)
#endif


Writing code/cuda_check.cuh


In [3]:
%%writefile code/TimingGPU.cuh
#ifndef TIMING_GPU_CUH
#define TIMING_GPU_CUH
#include <cuda_runtime.h>
class TimingGPU {
    cudaEvent_t start_;
    cudaEvent_t stop_;
    bool started_;
public:
    TimingGPU();
    ~TimingGPU();
    TimingGPU(const TimingGPU&) = delete;
    TimingGPU& operator=(const TimingGPU&) = delete;
    void StartCounter();
    float GetCounter(); // milliseconds
};
#endif


Writing code/TimingGPU.cuh


In [4]:
%%writefile code/TimingGPU.cu
#include "TimingGPU.cuh"
#include "cuda_check.cuh"
#include <cstdio>
#include <cstdlib>

TimingGPU::TimingGPU() : started_(false) {
    CUDA_CHECK(cudaEventCreate(&start_));
    CUDA_CHECK(cudaEventCreate(&stop_));
}
TimingGPU::~TimingGPU() {
    CUDA_CHECK(cudaEventDestroy(start_));
    CUDA_CHECK(cudaEventDestroy(stop_));
}
void TimingGPU::StartCounter() {
    CUDA_CHECK(cudaEventRecord(start_));
    started_ = true;
}
float TimingGPU::GetCounter() {
    if (!started_) {
        std::fprintf(stderr, "Call StartCounter before GetCounter.\n");
        std::exit(EXIT_FAILURE);
    }
    CUDA_CHECK(cudaEventRecord(stop_));
    CUDA_CHECK(cudaEventSynchronize(stop_));
    float milliseconds = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&milliseconds, start_, stop_));
    started_ = false;
    return milliseconds;
}


Writing code/TimingGPU.cu


## Kernels and host program

The direct kernel has coalesced loads and strided stores. In the tiled kernel, a block copies a `32×32` tile to shared memory, synchronizes, and reads it with transposed indices before writing the output. The extra column in `tile[32][33]` avoids a shared-memory bank conflict during that transposed read. The kernel supports dimensions that are not multiples of 32.

In [5]:
%%writefile code/chapter02_matrix_transpose.cu
#include "cuda_check.cuh"
#include "TimingGPU.cuh"
#include <cmath>
#include <cstdio>
#include <cstdlib>

constexpr int TILE = 32;
constexpr int BLOCK_ROWS = 8;

__global__ void transpose_direct(const float *input, float *output, int n) {
    const int col = blockIdx.x * TILE + threadIdx.x;
    for (int j = 0; j < TILE; j += BLOCK_ROWS) {
        const int row = blockIdx.y * TILE + threadIdx.y + j;
        if (row < n && col < n)
            output[col * n + row] = input[row * n + col];
    }
}

__global__ void transpose_shared(const float *input, float *output, int n) {
    __shared__ float tile[TILE][TILE + 1];
    const int col = blockIdx.x * TILE + threadIdx.x;
    const int row0 = blockIdx.y * TILE + threadIdx.y;

    for (int j = 0; j < TILE; j += BLOCK_ROWS) {
        const int row = row0 + j;
        tile[threadIdx.y + j][threadIdx.x] =
            (row < n && col < n) ? input[row * n + col] : 0.0f;
    }
    __syncthreads();

    const int out_col = blockIdx.y * TILE + threadIdx.x;
    const int out_row0 = blockIdx.x * TILE + threadIdx.y;
    for (int j = 0; j < TILE; j += BLOCK_ROWS) {
        const int out_row = out_row0 + j;
        if (out_row < n && out_col < n)
            output[out_row * n + out_col] = tile[threadIdx.x][threadIdx.y + j];
    }
}

void launch_transpose(bool use_shared, const float *input, float *output,
                      int n) {
    const dim3 block(TILE, BLOCK_ROWS);
    const dim3 grid((n + TILE - 1) / TILE, (n + TILE - 1) / TILE);
    if (use_shared)
        transpose_shared<<<grid, block>>>(input, output, n);
    else
        transpose_direct<<<grid, block>>>(input, output, n);
    CUDA_CHECK(cudaGetLastError());
}

void run_case(int n, bool print_matrix, int repeats) {
    const size_t bytes = size_t(n) * n * sizeof(float);
    float *h_input = (float*)std::malloc(bytes);
    float *h_cpu = (float*)std::malloc(bytes);
    float *h_direct = (float*)std::malloc(bytes);
    float *h_shared = (float*)std::malloc(bytes);
    float *d_input = nullptr, *d_direct = nullptr, *d_shared = nullptr;
    if (!h_input || !h_cpu || !h_direct || !h_shared) {
        std::fprintf(stderr, "Host allocation failed.\n");
        std::exit(EXIT_FAILURE);
    }
    for (int row = 0; row < n; ++row) {
        for (int col = 0; col < n; ++col) {
            const float value = float((row * 17 + col * 3) % 101) / 10.0f;
            h_input[row * n + col] = value;
            h_cpu[col * n + row] = value;
        }
    }
    CUDA_CHECK(cudaMalloc((void**)&d_input, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_direct, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_shared, bytes));
    CUDA_CHECK(cudaMemcpy(d_input, h_input, bytes, cudaMemcpyHostToDevice));

    launch_transpose(false, d_input, d_direct, n);
    launch_transpose(true, d_input, d_shared, n);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_direct, d_direct, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_shared, d_shared, bytes, cudaMemcpyDeviceToHost));
    for (int i = 0; i < n * n; ++i) {
        if (std::fabs(h_direct[i] - h_cpu[i]) > 1.0e-6f ||
            std::fabs(h_shared[i] - h_cpu[i]) > 1.0e-6f) {
            std::fprintf(stderr, "CPU/GPU mismatch at linear index %d.\n", i);
            std::exit(EXIT_FAILURE);
        }
    }
    std::printf("N=%d: direct GPU vs CPU PASS; shared GPU vs CPU PASS\n", n);
    if (print_matrix) {
        std::printf("Input matrix:\n");
        for (int row = 0; row < n; ++row) {
            for (int col = 0; col < n; ++col)
                std::printf("%6.1f", h_input[row * n + col]);
            std::printf("\n");
        }
        std::printf("Transposed matrix (CPU, direct GPU, shared GPU):\n");
        for (int row = 0; row < n; ++row) {
            for (int col = 0; col < n; ++col) {
                const int i = row * n + col;
                std::printf("%5.1f/%5.1f/%5.1f  ", h_cpu[i], h_direct[i], h_shared[i]);
            }
            std::printf("\n");
        }
    } else {
        TimingGPU timer;
        float elapsed[2];
        for (int method = 0; method < 2; ++method) {
            for (int k = 0; k < 10; ++k)
                launch_transpose(method == 1, d_input,
                                 method ? d_shared : d_direct, n);
            CUDA_CHECK(cudaDeviceSynchronize());
            timer.StartCounter();
            for (int k = 0; k < repeats; ++k)
                launch_transpose(method == 1, d_input,
                                 method ? d_shared : d_direct, n);
            elapsed[method] = timer.GetCounter() / repeats;
        }
        std::printf("N=%d, average of %d kernel launches\n", n, repeats);
        std::printf("Direct transpose:       %.6f ms/launch\n", elapsed[0]);
        std::printf("Shared-memory transpose: %.6f ms/launch\n", elapsed[1]);
        std::printf("Direct/shared time ratio: %.3f\n", elapsed[0] / elapsed[1]);
        std::printf("Timing covers kernels only; allocation, copies, and validation are excluded.\n");
    }
    CUDA_CHECK(cudaFree(d_input));
    CUDA_CHECK(cudaFree(d_direct));
    CUDA_CHECK(cudaFree(d_shared));
    std::free(h_input); std::free(h_cpu); std::free(h_direct); std::free(h_shared);
}

int main() {
    run_case(5, true, 0);          // prints and validates a non-tile-sized case
    run_case(2048, false, 100);    // compares kernel time on the selected GPU
    return 0;
}


Writing code/chapter02_matrix_transpose.cu


## Compile and run

The first case prints the small input and the CPU, direct-GPU, and shared-GPU transposes. The second case validates a larger matrix and prints average kernel time for each implementation. Timing excludes allocation and host-device copies. The ratio is hardware-dependent; a value above 1 means the shared-memory kernel was faster on this GPU.

In [6]:
%%bash
set -eu
nvcc -O2 -std=c++14 -Icode code/chapter02_matrix_transpose.cu code/TimingGPU.cu -o matrix_transpose
./matrix_transpose


N=5: direct GPU vs CPU PASS; shared GPU vs CPU PASS
Input matrix:
   0.0   0.3   0.6   0.9   1.2
   1.7   2.0   2.3   2.6   2.9
   3.4   3.7   4.0   4.3   4.6
   5.1   5.4   5.7   6.0   6.3
   6.8   7.1   7.4   7.7   8.0
Transposed matrix (CPU, direct GPU, shared GPU):
  0.0/  0.0/  0.0    1.7/  1.7/  1.7    3.4/  3.4/  3.4    5.1/  5.1/  5.1    6.8/  6.8/  6.8  
  0.3/  0.3/  0.3    2.0/  2.0/  2.0    3.7/  3.7/  3.7    5.4/  5.4/  5.4    7.1/  7.1/  7.1  
  0.6/  0.6/  0.6    2.3/  2.3/  2.3    4.0/  4.0/  4.0    5.7/  5.7/  5.7    7.4/  7.4/  7.4  
  0.9/  0.9/  0.9    2.6/  2.6/  2.6    4.3/  4.3/  4.3    6.0/  6.0/  6.0    7.7/  7.7/  7.7  
  1.2/  1.2/  1.2    2.9/  2.9/  2.9    4.6/  4.6/  4.6    6.3/  6.3/  6.3    8.0/  8.0/  8.0  
N=2048: direct GPU vs CPU PASS; shared GPU vs CPU PASS
N=2048, average of 100 kernel launches
Direct transpose:       0.587940 ms/launch
Shared-memory transpose: 0.163880 ms/launch
Direct/shared time ratio: 3.588
Timing covers kernels only; allocatio